# 01 Data validation

> **Synthetic data, sampled simulation.** 768 simulated runs compared with 19,708 scheduled runs (3.9%). Not IRCTC or Indian Railways network-wide data. No forecasts. Associations are not causes.

**Purpose.** Load the FULL validated cleaned dataset (Phase 2 output, `data/processed/cleaned`), document what is in it, and validate it with independent Python checks. The 48-run Excel sample is not used. The files are only read, never written.

The same logic runs in `python/src/run_phase5.py`; this notebook shows it step by step.

In [1]:
# Setup: find the project root, make python/src importable, load shared helpers (the data is only READ)
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "data" / "processed" / "cleaned").exists())
sys.path.insert(0, str(ROOT / "python" / "src"))
import pandas as pd, numpy as np
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from analytics import data_loader as dl, validation, cleaning, kpi, eda, SYNTHETIC_NOTE
from analytics import statistics as stats
print(SYNTHETIC_NOTE)

Synthetic data, sampled simulation (768 simulated runs of 19,708 scheduled). Not IRCTC or Indian Railways network-wide data; no forecasts.


## 1. Sources, row counts, types and date ranges

In [2]:
ds = dl.load_dataset()
display(dl.source_register(ds.tables))
display(dl.profile_tables(ds.tables).drop(columns='dtypes'))
print('Total rows:', sum(len(t) for t in ds.tables.values()))

,source,rows,role
0,data/processed/cleaned/stations.csv,32,primary analytical source (full cleaned dataset)
1,data/processed/cleaned/routes.csv,40,primary analytical source (full cleaned dataset)
2,data/processed/cleaned/trains.csv,52,primary analytical source (full cleaned dataset)
3,data/processed/cleaned/train_stops.csv,203,primary analytical source (full cleaned dataset)
4,data/processed/cleaned/calendar.csv,975,primary analytical source (full cleaned dataset)
5,data/processed/cleaned/seat_capacity.csv,2285,primary analytical source (full cleaned dataset)
6,data/processed/cleaned/train_operations.csv,768,primary analytical source (full cleaned dataset)
7,data/processed/cleaned/bookings.csv,201182,primary analytical source (full cleaned dataset)
8,data/processed/cleaned/booking_passengers.csv,405200,primary analytical source (full cleaned dataset)
9,data/processed/cleaned/cancellations.csv,43923,primary analytical source (full cleaned dataset)


,table,rows,columns,null_cells,date_range,data_origin
0,stations,32,7,0,,SYNTHETIC
1,routes,40,6,0,,SYNTHETIC
2,trains,52,7,0,,SYNTHETIC
3,train_stops,203,8,104,,SYNTHETIC
4,calendar,975,17,1732,calendar_date: 2024-05-01 to 2026-12-31,SYNTHETIC
5,seat_capacity,2285,10,0,journey_date: 2024-09-01 to 2026-08-31,SYNTHETIC
6,train_operations,768,6,23,run_date: 2024-09-01 to 2026-08-31,SYNTHETIC
7,bookings,201182,13,0,journey_date: 2024-09-01 to 2026-08-31; bookin...,SYNTHETIC
8,booking_passengers,405200,9,325020,,SYNTHETIC
9,cancellations,43923,10,0,cancellation_ts: 2024-05-31 to 2026-08-31,SYNTHETIC


Total rows: 654660


## 2. Data types

Explicit dtypes are applied at load time (integers, floats, booleans, parsed dates and timestamps, nullable integers where NULL is documented).

In [3]:
display(pd.DataFrame({t: df.dtypes.astype(str).value_counts() for t, df in ds.tables.items()}).fillna(0).astype(int).T)

,Int64,bool,datetime64[us],float64,int64,str
stations,0,0,0,0,1,6
routes,0,0,0,1,3,2
trains,0,0,0,0,2,5
train_stops,0,0,0,1,4,3
calendar,0,3,1,0,7,6
seat_capacity,0,0,1,0,7,2
train_operations,1,0,1,0,3,1
bookings,0,0,2,1,6,4
booking_passengers,1,0,0,1,3,4
cancellations,0,0,1,3,3,3


## 3. Relationships between tables

Every foreign key is checked for orphan rows (NULL children, such as `cancellation_id` on Confirmed passengers, are allowed).

In [4]:
display(dl.relationship_report(ds.tables))

,child,parent,child_rows,orphans
0,routes.origin_station_id,stations.station_id,40,0
1,routes.destination_station_id,stations.station_id,40,0
2,trains.route_id,routes.route_id,52,0
3,train_stops.train_id,trains.train_id,203,0
4,train_stops.station_id,stations.station_id,203,0
5,seat_capacity.train_id,trains.train_id,2285,0
6,seat_capacity.journey_date,calendar.calendar_date,2285,0
7,train_operations.train_id,trains.train_id,768,0
8,train_operations.run_date,calendar.calendar_date,768,0
9,bookings.train_id,trains.train_id,201182,0


## 4. Documented gaps (retained, never imputed)

In [5]:
display(validation.documented_gaps(ds.tables))

,item,rows,treatment
0,train_operations.arrival_delay_min is NULL,23,retained as NULL; excluded from delay KPIs (de...
1,booking_passengers.age_band = 'Unknown',1010,retained as 'Unknown'; no age imputed; not use...
2,booking_passengers.cancellation_id is NULL,325020,expected: Confirmed and RAC passengers have no...


## 5. Validation checks

Row counts are compared with the Phase 2 validated results (`reports/data_quality/row_movement.csv`). Each check returns the number of violating records; 0 means PASS.

In [6]:
val = validation.run_validation(ds.tables, ds.reference)
display(val.groupby('category', sort=False).agg(checks=('check_id', 'size'), passed=('passed', 'sum'), violations=('violations', 'sum')))
print(f'{int(val.passed.sum())} of {len(val)} checks passed')
display(val[~val.passed])

,checks,passed,violations
category,,,
ROW COUNTS,10,10,0
DUPLICATES,15,15,0
MISSING VALUES,10,10,0
DATES,11,11,0
CATEGORIES,11,11,0
AMOUNTS,4,4,0
CAPACITY,3,3,0
OCCUPANCY BOUNDS,4,4,0
RECONCILIATION,8,8,0


111 of 111 checks passed


,check_id,category,description,violations,passed


## 6. Every check

(The report `reports/phase5_python_validation.txt` lists them with descriptions.)

In [7]:
display(val[['check_id', 'category', 'violations', 'passed']].head(40))

,check_id,category,violations,passed
0,ROWS-stations,ROW COUNTS,0,True
1,ROWS-routes,ROW COUNTS,0,True
2,ROWS-trains,ROW COUNTS,0,True
3,ROWS-train_stops,ROW COUNTS,0,True
4,ROWS-calendar,ROW COUNTS,0,True
5,ROWS-seat_capacity,ROW COUNTS,0,True
6,ROWS-train_operations,ROW COUNTS,0,True
7,ROWS-bookings,ROW COUNTS,0,True
8,ROWS-booking_passengers,ROW COUNTS,0,True
9,ROWS-cancellations,ROW COUNTS,0,True


## 7. Reference outputs from Phase 4

The PostgreSQL layer's own 72 data-quality checks (built and run on PostgreSQL 16; **not yet run on PostgreSQL 18.4 / Windows**).

In [8]:
p4 = ds.reference['phase4_checks']
print(f"{int(p4.passed.sum())} of {len(p4)} PostgreSQL checks passed in the Phase 4 build")

72 of 72 PostgreSQL checks passed in the Phase 4 build
